# 量子位相推定

ユニタリ $U$ とその固有状態 $\lvert \psi \rangle$ が準備できているとき、固有値 $\lambda = e^{-i\alpha}$ の位相 $\alpha$ を求める量子位相推定について説明します。

$$
U \lvert \psi \rangle = e^{-i\alpha} \lvert \psi \rangle
$$

## 量子位相推定の量子回路
量子位相推定アルゴリズムの全体像は次の通りです。

1. 固有ベクトルとしての量子状態の準備。
2. 固有値を抽出し、量子状態に埋め込む(位相キックバック)。
3. 量子状態を変換し、固有値を測定ビット列として取得する(逆量子フーリエ変換)。



```
                         step2                   step3

|0> ----H----------------------*-------iQFT---
|0> ----H--------------*-------|-------iQFT---
|0> ----H--------*-----|-------|-------iQFT---
|0> ----H--*-----|-----|-------|-------iQFT---
                  |       |       |         |
                  |       |       |         |
|ψ> -------U1--U2---U4-- -U2n------------
step1
```

## 位相キックバック

ここでは位相キックバック法を振り返ります。

固有ベクトル $\lvert \psi \rangle$ とそれに対応する固有値 $e^{2\pi i \phi}$ を持つユニタリに対して、その制御ユニタリ回路を導入します。固有値は制御ビットの $\lvert 1\rangle$ の係数として埋め込むことができることを見ていきます。

このユニタリに対する固有値方程式は次の通りです。

$$
U\lvert \psi \rangle = e^{2\pi i \phi} \lvert \psi \rangle
$$

量子状態 $\lvert \psi \rangle$ が準備できているとすると、制御ユニタリ回路によって位相を抽出することができます。

まず、初期状態から、結果を取り出したい量子ビットに対して Hadamard ゲートを使って重ね合わせ状態を作ります。

$$
\lvert 0 \rangle \lvert \psi \rangle \rightarrow \frac{\lvert 0\rangle + \lvert 1 \rangle}{\sqrt{2}} \lvert \psi \rangle = \frac{\lvert 0\rangle \lvert \psi \rangle + \lvert 1 \rangle \lvert \psi \rangle}{\sqrt{2}}
$$

次に、制御ユニタリ回路を導入することで、制御量子ビットが $\lvert 1\rangle$ の状態にのみユニタリが作用します。

$$
\frac{\lvert 0\rangle \lvert \psi \rangle + \lvert 1 \rangle U \lvert \psi \rangle}{\sqrt{2}}
$$

固有値方程式を使うと、固有値を係数として取り出すことができます。

$$
\frac{\lvert 0\rangle \lvert \psi \rangle + \lvert 1 \rangle e^{2\pi i \phi} \lvert \psi \rangle}{\sqrt{2}} = \frac{\lvert 0\rangle \lvert \psi \rangle +  e^{2\pi i \phi} \lvert 1 \rangle \lvert \psi \rangle}{\sqrt{2}} = \frac{\lvert 0\rangle +  e^{2\pi i \phi} \mid 1 \rangle}{\sqrt{2}} \lvert \psi \rangle
$$


次に、

$$
\frac{1}{\sqrt{2^n}}\sum_{k=0}^{2^n-1} e^{i2\pi k\phi}\lvert k \rangle
$$

のように、取り出したい桁に対応する $k$ を導入すれば良いのですが、これは回転角に対応しており、固有値を $k$ 回かけるということをすると、自然と対応させることができます。つまり $U^k$ のように同じユニタリ操作を $k$ 回実行すれば良いことになります。ということで、$k$ 回 Controlled-Unitary 操作を行うことで求めることができます。

$$
\frac{\lvert 0\rangle +  U^k \lvert 1 \rangle}{\sqrt{2}} \lvert \psi \rangle = \frac{\lvert 0\rangle +  e^{2\pi i k \phi} \lvert 1 \rangle}{\sqrt{2}} \lvert \psi \rangle
$$

これにより、対応する桁 $k$ に対して $k$ 回の制御付きユニタリゲートをかけることで実行が終了します。

## 量子フーリエ変換

ここでは量子フーリエ変換についても振り返ります。

量子フーリエ変換は、2進数のビット列を、対応する位相を持つ量子状態に変換することができます。
量子フーリエ変換の逆回路である逆量子フーリエ変換を使うことで、上記の位相キックバックによって転写された位相を、ビット列として書き出すことができます。

$$
QFT:\lvert x \rangle \mapsto \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} \omega_n^{xk}\lvert k\rangle
$$

<!--
Assuming $\omega_n = e^{\frac{2\pi i}{N}}$,

$$
{F_N=
\frac{1}{\sqrt{N}}
\left[
 \begin{array}{rrrr}
  1 & 1 & 1 & \cdots &1\\
  1 & \omega_n&\omega_n^2&\cdots&\omega_n^{N-1}\\
  1 & \omega_n^2&\omega_n^4&\cdots&\omega_n^{2(N-1)}\\
  1 & \omega_n^3&\omega_n^6&\cdots&\omega_n^{3(N-1)}\\
  \vdots&\vdots&\vdots&&\vdots\\
  1 & \omega_n^{N-1}&\omega_n^{2(N-1)}&\cdots&\omega_n^{(N-1)(N-1)}
 \end{array}
\right]
}
$$ -->

ビット列 $x_n$ から $x_1$ までを入力すると、出力される量子状態は入力ビット列に対応した位相を持ちます。

$$
QFT(\lvert x_n,x_{n-1},…,x_1 \rangle) = \frac{1}{\sqrt{N}}(\lvert 0 \rangle + e^{2\pi i [0.x_n]} \lvert 1 \rangle) \otimes … \otimes (\lvert 0 \rangle + e^{2\pi i [0.x_1x_2…x_n]} \lvert 1 \rangle)
$$

$$[0.x_1x_2…] = \frac{x_1}{2}+\frac{x_2}{2^2}+…$$

入力状態(ビット列)を桁ごとにずらしたものが、出力量子状態の各量子ビットの相対位相として符号化されます。ただし、各量子ビットの $\lvert 1\rangle$ の係数の絶対値はすべて $1/\sqrt{N}$ であるため、個々の量子ビットをそのまま測定すると、ちょうど 0 と 1 が 50% ずつ観測されてしまいます。

## 位相キックバック + 逆量子フーリエ変換

量子位相推定では、先ほど説明した位相キックバックを適用して、次の状態を準備します。

$$
\frac{1}{\sqrt{2^n}}\sum_{k=0}^{2^n-1} e^{i2\pi k\phi}\lvert k \rangle
$$

$\phi$ はユニタリの固有値であり、これが求めたいものです。

このような状態を準備できれば、逆量子フーリエ変換を用いることで、$\phi$ を2進数表記したビット列からなる量子状態 $\lvert \phi_n,\phi_{n-1},...,\phi_1\rangle$ に変換することができます。
したがって、最後に測定を行えば、$\phi$ を取り出すことができます。

各項において、元の位相 $\phi$ は $k$ 倍されていますが、これは固有値を $k$ 回かけることと同じです。
したがって、同じユニタリ操作を $U^k$ として $k$ 回実行すればよいことになります。言い換えると、Controlled-Unitary 操作を $k$ 回実行することで達成できます。

$$
\frac{\mid 0\rangle +  U^k \mid 1 \rangle}{\sqrt{2}} \mid \psi \rangle = \frac{\mid 0\rangle +  e^{2\pi i k \phi} \mid 1 \rangle}{\sqrt{2}} \mid \psi \rangle
$$

このように、対応する $k$ 回の制御付きユニタリゲートを各量子ビットに適用することで、上記の状態を準備することができます。

## Z ゲートの位相を推定する

演習をしてみましょう。まず、Z ゲートをユニタリとして準備します。

$$
Z = \begin{pmatrix}
1&0\\
0&-1
\end{pmatrix}
$$

まず最初に、手計算で答えを確認しておきます。
固有方程式を計算すると

$$
det\begin{pmatrix}
1-\lambda&0\\
0&-1-\lambda
\end{pmatrix} = 0
$$

となり、固有値は $\lambda = 1,-1$ であることがわかります。
固有ベクトルは以下の通りです。

$$
\begin{pmatrix}
1\\
0
\end{pmatrix},
\begin{pmatrix}
0\\
1
\end{pmatrix}
$$

それでは確認してみましょう。

## blueqat のインストール

In [ ]:
!pip install git+https://github.com/blueqat/blueqatSDK

## 回路の概要

回路の概要は以下の通りです。

```
|0> ----H--*--iQFT--M
                  |
|0> ------- Z--------
```

まず、0番目と1番目の2つの量子ビットを準備します。どちらも初期状態は $\lvert 0\rangle$ です。

1. 1番目の量子ビットに固有状態を準備する。
2. 0番目の量子ビットに Hadamard ゲートを作用させ、重ね合わせ状態を作る。
3. 制御ユニタリとして CZ ゲートを適用し、位相を0番目の量子ビットにキックバックする。
4. 逆量子フーリエ変換を行い、測定して位相を取り出す。

まず、固有状態として $\lvert 0\rangle$ を準備します。つまり、初期状態に対して何もしません。

そして、1量子ビットの量子フーリエ変換は Hadamard ゲートと等価です。
(Hadamard 行列はエルミートであるため、この場合の逆量子フーリエ変換も Hadamard ゲートになります)

したがって、実装は次のようになります。

In [ ]:
from blueqat import Circuit

Circuit().h[0].cz[0,1].h[0].m[:].run(shots=100)

したがって $\phi = 0.0$ です。

求めたい固有値は次の通りです。

$$
e^{2\pi i \phi} = e^{2\pi i \cdot 0} = e^0 = 1
$$

次に、準備する固有状態を $\lvert 1 \rangle$ に設定してみましょう。

```
|0> --H--*--iQFT--M
               |
|0> --X--Z--------
```

In [ ]:
Circuit().x[1].h[0].cz[0,1].h[0].m[:].run(shots=100)

これで $\phi = 1/2 = 0.5$ となりました。

求めたい固有値は次の通りです。

$$
e^{2\pi i \cdot 0.5} = -1
$$

## X ゲートの位相を推定する場合

X ゲートは以下のような行列です。

$$
X =
\begin{pmatrix}
0&1\\
1&0
\end{pmatrix}
$$

まず、以下の固有ベクトルを考えます。固有値は 1 です。

$$
\mid \psi \rangle =
\begin{pmatrix}
1\\
1
\end{pmatrix}
$$

回路は以下の通りです。

```
|0> --H--*--H--M
         　　　　　　|
|0> --H--X-----
```

In [ ]:
Circuit(2).h[:].cx[0,1].h[0].m[0].run(shots=100)

これは $\phi = 0.0$ であり、固有値は次の通りです。

$$
\lambda = e^0=1
$$

次に、以下の固有ベクトルを持つ場合を考えます。固有値は -1 です。

$$
\mid \psi \rangle =
\begin{pmatrix}
1\\
-1
\end{pmatrix}
$$


```
|0> --H---*--H--M
          　　　　　　|
|0> --HZ--X-----
```

In [ ]:
Circuit(2).h[:].z[1].cx[0,1].h[0].m[0].run(shots=100)

したがって $\phi = 0.5$ であり、固有値は次の通りです。

$$
\lambda = e^{2\pi i \cdot0.5}=-1
$$